# Case study: regression on EBTI decisions and Telco churn

Session 6 · Introduction to machine learning with regression. Author: course team, licence CC-BY-4.0.

The notebook follows the three practice tasks of the session:

1. **Block 1**: map the leaderboard task (predict the HS heading of a Binding Tariff Information decision) to the ten lifecycle steps; split the data and model the length of the description of goods (log scale) with linear regression; evaluate with MAE, RMSE and R².
2. **Block 2**: training versus test error for increasing model complexity; least squares versus robust regression (Huber, quantile) on the number of days that early-invalidated decisions stayed valid, a skewed target with gross errors.
3. **Block 3**: predict churn probability for the IBM Telco customers with logistic regression and interpret the coefficients.

The regression targets of blocks 1 and 2 are chosen for practice; they are not the leaderboard task. Cells marked **Exercise** are for you. Theory: `../theory/01-lifecycle-and-linear-regression.md` to `03-logistic-regression.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import HuberRegressor, LinearRegression, LogisticRegression
from sklearn.metrics import (confusion_matrix, mean_absolute_error, r2_score,
                             root_mean_squared_error)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet")
decisions = decisions.merge(nomenclature[["heading", "section"]], on="heading")

decisions["log_chars"] = np.log(decisions["description"].str.len())
decisions["year"] = decisions["start_date"].dt.year - 2017                  # 0 = 2017
decisions["n_lines"] = decisions["description"].str.count("\n") + 1
decisions["has_code"] = decisions["description"].str.contains("<CODE>", regex=False).astype(int)
top = decisions["language"].value_counts().index[:8]                         # rare languages -> other
decisions["language"] = decisions["language"].where(decisions["language"].isin(top), "other")
print(decisions.shape)

## Block 1 · Lifecycle, split and linear regression

### 1.1 The heading-classification task in ten steps

**Exercise.** Complete the table for the leaderboard task (predict the four-digit HS heading from the description of goods and the request's metadata; metric accuracy, macro-F1 alongside). Think about which columns of the training data are not available at prediction time.

| Step | What it means for the heading task | Session |
|---|---|---|
| 01 Problem definition | target = heading (1,114 classes); metric = accuracy, macro-F1 alongside; baseline = always 3926 (accuracy 0.041) | S6 |
| 02 Data collection | … (EBTI export; test set contains only description, country, language, date) | S2, S3 |
| 03 Data cleaning and preprocessing | … | S4 |
| 04 Exploratory data analysis | … | S5 |
| 05 Feature engineering and selection | … | |
| 06 Model selection | … | |
| 07 Model training | … | |
| 08 Model evaluation and tuning | … | |
| 09 Model deployment | … | |
| 10 Model monitoring and maintenance | … | |

### 1.2 Train–test split

Target: `log_chars` = log(number of characters of the description). The raw length is right-skewed (Session 5), and on the log scale a coefficient reads as a percentage difference.

In [ ]:
features = ["year", "language", "section", "n_lines", "has_code"]
train, test = train_test_split(decisions, test_size=0.2, random_state=42)
print(len(train), len(test))
train[["log_chars", "year", "n_lines", "has_code"]].describe().round(2)

### 1.3 Simple and multiple linear regression

In [ ]:
fit_simple = smf.ols("log_chars ~ year", data=train).fit()
fit_multi = smf.ols('log_chars ~ year + C(language, Treatment("de")) + C(section)', data=train).fit()
print(fit_simple.params.round(4))
print("R2 simple:", round(fit_simple.rsquared, 3), " R2 multiple:", round(fit_multi.rsquared, 3))
coef = fit_multi.params.filter(like="language").rename(lambda n: n.split("[T.")[1][:-1])
pd.DataFrame({"coef (log scale)": coef, "ratio to German": np.exp(coef)}).round(3)

**Exercise.** Interpret the coefficients of `year` and of French in one sentence each ("holding the other features fixed, …"; on the log scale e^b − 1 is the percentage difference). Why does the year coefficient change when language and section are added?

### 1.4 Metrics on the test set, compared with the baseline

`n_lines` is computed from the description itself, so it nearly determines the length: it is shown to make a point about features that are almost the target, not as a sensible model.

In [ ]:
X = pd.get_dummies(decisions[features], columns=["language", "section"], drop_first=True, dtype=float)
X_train, X_test = X.loc[train.index], X.loc[test.index]
lang_cols = ["year"] + [c for c in X if c.startswith("language_")]
sect_cols = lang_cols + [c for c in X if c.startswith("section_")]

def evaluate(name, model, cols, target="log_chars"):
    model.fit(X_train[cols], train[target])
    pred_train, pred_test = model.predict(X_train[cols]), model.predict(X_test[cols])
    return {"model": name,
            "MAE": mean_absolute_error(test[target], pred_test),
            "RMSE": root_mean_squared_error(test[target], pred_test),
            "R2 test": r2_score(test[target], pred_test),
            "R2 train": r2_score(train[target], pred_train)}

results = pd.DataFrame([
    evaluate("baseline (mean)", DummyRegressor(), ["year"]),
    evaluate("simple: year", LinearRegression(), ["year"]),
    evaluate("year + language", LinearRegression(), lang_cols),
    evaluate("year + language + section", LinearRegression(), sect_cols),
    evaluate("... + number of lines", LinearRegression(), sect_cols + ["n_lines", "has_code"]),
]).set_index("model").round(3)
results

In [ ]:
model = LinearRegression().fit(X_train[sect_cols], train["log_chars"])
fitted = model.predict(X_test[sect_cols])
resid = test["log_chars"] - fitted
fig, ax = plt.subplots(figsize=(6.5, 4), layout="constrained")
ax.scatter(fitted, resid, s=4, alpha=0.3, color="0.4")
ax.axhline(0, color="black", lw=1)
ax.set(xlabel="fitted log(characters)", ylabel="residual", title="Residuals on the test set");
print(test.loc[resid.nsmallest(3).index, ["language", "description"]].to_string()[:600])

**Exercise.** Describe the residual plot. Where are the largest negative residuals, and what do the three printed descriptions have in common?

## Block 2 · Model complexity and robust regression

### 2.1 Training versus test error for increasing complexity

First on toy data (20 points from a cosine curve), then on the case study.

In [ ]:
rng = np.random.default_rng(0)
x_tr = np.sort(rng.uniform(0, 1, 20)).reshape(-1, 1)
y_tr = np.cos(1.5 * np.pi * x_tr.ravel()) + rng.normal(0, 0.15, 20)
x_te = rng.uniform(0, 1, 1000).reshape(-1, 1)
y_te = np.cos(1.5 * np.pi * x_te.ravel()) + rng.normal(0, 0.15, 1000)

rows = []
for degree in range(1, 16):
    m = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_tr, y_tr)
    rows.append({"degree": degree,
                 "train RMSE": root_mean_squared_error(y_tr, m.predict(x_tr)),
                 "test RMSE": root_mean_squared_error(y_te, m.predict(x_te))})
curve = pd.DataFrame(rows).set_index("degree")
ax = curve.plot(logy=True, marker="o", figsize=(6.5, 3.8), title="Toy data: training vs test error")
ax.set_ylabel("RMSE (log scale)")
curve.round(3).T

The case-study target for block 2: decisions that were invalidated **before** their normal three-year expiry (they have an invalidation reason code), and the number of days they stayed valid. One feature: the start date in years since 2017. Many of these decisions end on fixed dates, so the relationship is not a straight line.

In [ ]:
early = decisions[decisions["invalidation_reason"].notna()].copy()
early["days_valid"] = (early["end_date"] - early["start_date"]).dt.days
early["start_year"] = (early["start_date"] - pd.Timestamp("2017-01-01")).dt.days / 365.25
print(len(early), "early-invalidated decisions;", (early["days_valid"] < 0).sum(), "with negative duration")
print("most frequent end dates:", early["end_date"].dt.date.value_counts().head(3).to_dict())
clean = early[early["days_valid"] >= 0]
e_train, e_test = train_test_split(clean, test_size=0.2, random_state=42)

rows = []
for degree in [1, 2, 3, 5, 8, 12]:
    m = make_pipeline(StandardScaler(), PolynomialFeatures(degree), LinearRegression())
    m.fit(e_train[["start_year"]], e_train["days_valid"])
    rows.append({"degree": degree,
                 "train RMSE": root_mean_squared_error(e_train["days_valid"], m.predict(e_train[["start_year"]])),
                 "test RMSE": root_mean_squared_error(e_test["days_valid"], m.predict(e_test[["start_year"]]))})
pd.DataFrame(rows).set_index("degree").round(1)

**Exercise.** Repeat the experiment with only 60 training decisions (`e_train.sample(60, random_state=0)`). At which degree does the test error start to rise? What does this say about flexibility relative to the amount of data? Then plot `days_valid` against `start_year`: which diagonal lines do you see, and what calendar events could cause them?

In [ ]:
# Exercise: complexity with 60 training decisions
# small = e_train.sample(60, random_state=0)
# ...

### 2.2 Least squares versus robust regression

Now keep the 77 decisions with negative durations: their end date is a placeholder in the year 1900. They are gross errors in y, as in the theory figure.

In [ ]:
print(early.loc[early["days_valid"] < 0, ["invalidation_reason"]].value_counts().to_dict(),
      early.loc[early["days_valid"] < 0, "end_date"].dt.year.unique())
r_train, r_test = train_test_split(early, test_size=0.2, random_state=42)
r_test = r_test[r_test["days_valid"] >= 0]                     # evaluate on plausible rows only

rows = []
for name, fit in {
    "OLS (mean)": smf.ols("days_valid ~ start_year", r_train).fit(),
    "OLS, placeholders removed": smf.ols("days_valid ~ start_year", r_train[r_train["days_valid"] >= 0]).fit(),
    "median (q=0.5)": smf.quantreg("days_valid ~ start_year", r_train).fit(q=0.5),
    "q=0.9": smf.quantreg("days_valid ~ start_year", r_train).fit(q=0.9),
}.items():
    pred = fit.predict(r_test)
    rows.append({"model": name, "intercept": fit.params["Intercept"], "slope": fit.params["start_year"],
                 "MAE": mean_absolute_error(r_test["days_valid"], pred),
                 "RMSE": root_mean_squared_error(r_test["days_valid"], pred)})

huber = HuberRegressor(max_iter=1000).fit(r_train[["start_year"]], r_train["days_valid"])
pred = huber.predict(r_test[["start_year"]])
rows.append({"model": "Huber", "intercept": huber.intercept_, "slope": huber.coef_[0],
             "MAE": mean_absolute_error(r_test["days_valid"], pred),
             "RMSE": root_mean_squared_error(r_test["days_valid"], pred)})
pd.DataFrame(rows).set_index("model").round(1)

In [ ]:
grid = pd.DataFrame({"start_year": np.linspace(0, 7, 50)})
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.scatter(clean["start_year"], clean["days_valid"], s=4, alpha=0.3, color="0.6", label="decisions")
for q, colour in [(0.5, "#009E73"), (0.9, "#0072B2")]:
    ax.plot(grid["start_year"], smf.quantreg("days_valid ~ start_year", r_train).fit(q=q).predict(grid),
            color=colour, lw=2, label=f"quantile {q}")
ax.plot(grid["start_year"], smf.ols("days_valid ~ start_year", r_train).fit().predict(grid),
        color="#D55E00", lw=2, ls="--", label="least squares, with placeholder rows")
ax.set(ylim=(-50, 1150), xlabel="start date (years since 2017)", ylabel="days valid",
       title="Early-invalidated decisions: mean, median and 90th percentile")
ax.legend(frameon=False, fontsize=8);

**Exercise.** (a) How much do the 77 placeholder rows move the least-squares line, and how much the median and Huber lines? (b) Which model would you use to tell a trader how long a *typical* early-invalidated decision lasts, and which to describe the decisions that last *longest*? (c) Why is it still better to remove the placeholder rows once you have found them?

## Block 3 · Logistic regression: Telco churn

The IBM Telco sample (7,043 customers) is downloaded from GitHub (about 1 MB).

In [ ]:
url = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
telco = pd.read_csv(url)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")   # 11 blanks -> NaN
telco["churn"] = (telco["Churn"] == "Yes").astype(int)
print(telco.shape, "churn rate:", round(telco["churn"].mean(), 3))
telco.groupby("Contract")["churn"].agg(["mean", "size"]).round(3)

In [ ]:
tr, te = train_test_split(telco, test_size=0.25, stratify=telco["churn"], random_state=0)
formula = ("churn ~ tenure + MonthlyCharges + SeniorCitizen + C(Contract) + C(InternetService)"
           " + C(PaymentMethod) + C(PaperlessBilling)")
logit = smf.logit(formula, data=tr).fit(disp=0)
coef = pd.DataFrame({"coef (log-odds)": logit.params, "odds ratio": np.exp(logit.params),
                     "OR low": np.exp(logit.conf_int()[0]), "OR high": np.exp(logit.conf_int()[1]),
                     "p": logit.pvalues})
coef.round(3)

**Exercise.** Interpret three coefficients as odds ratios (for example two-year contract, fibre optic, tenure per 12 months: `np.exp(12 * coef)`). Then translate one of them into probabilities for two typical customers with the cell below.

In [ ]:
typical = pd.DataFrame({
    "tenure": [3, 3], "MonthlyCharges": [80, 80], "SeniorCitizen": [0, 0],
    "Contract": ["Month-to-month", "Two year"], "InternetService": ["Fiber optic", "Fiber optic"],
    "PaymentMethod": ["Electronic check", "Electronic check"], "PaperlessBilling": ["Yes", "Yes"],
})
typical.assign(p_churn=logit.predict(typical).round(3))

### The same model in scikit-learn

In [ ]:
numeric = ["tenure", "MonthlyCharges", "SeniorCitizen"]
categorical = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]
pre = ColumnTransformer([("num", "passthrough", numeric),
                         ("cat", OneHotEncoder(drop="first"), categorical)])
clf = make_pipeline(pre, LogisticRegression(C=np.inf, max_iter=5000)).fit(tr, tr["churn"])
sk_coef = pd.Series(clf[-1].coef_[0], index=clf[0].get_feature_names_out())
sk_coef.round(3)

### A first look at predicted classes

In [ ]:
p = clf.predict_proba(te)[:, 1]
print("majority baseline accuracy:", round(1 - te["churn"].mean(), 3))
rows = []
for threshold in [0.5, 0.4, 0.3, 0.2]:
    pred = (p > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(te["churn"], pred).ravel()
    rows.append({"threshold": threshold, "accuracy": (pred == te["churn"]).mean(),
                 "TP (churners found)": tp, "FN (missed)": fn, "FP (false alarms)": fp, "TN": tn})
pd.DataFrame(rows).set_index("threshold").round(3)

**Exercise.** A retention call costs €10; a lost customer costs €200 of margin. Which threshold minimises the total cost on the test set (cost = 10 × (TP + FP) + 200 × FN)? Session 8 returns to this question with ROC curves.